# Silver Layer — Cleaning NJ Transit Data

This notebook cleans the bronze-layer NJ Transit tables: casts columns to proper types, normalizes line names, standardizes trip status, and writes curated tables to the `workspace.nj_transit_silver` schema.

In [0]:
# --- PySpark (work with imported data in the lakehouse) ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException

# --- Python standard libs for date/time handling ---
from datetime import datetime, date

In [0]:
# Load bronze transit_trips
df_trips = spark.table("workspace.nj_transit_bronze.transit_trips")

# Cast to proper types and trim whitespace from string columns
df_trips_cleaned = df_trips.select(
    F.to_date("date", "yyyy-MM-dd").alias("date"),
    F.trim(F.col("train_id")).alias("train_id"),
    F.col("stop_sequence").cast("double").cast("int").alias("stop_sequence"),
    F.trim(F.col("from")).alias("from"),
    F.col("from_id").cast("int").alias("from_id"),
    F.trim(F.col("to")).alias("to"),
    F.col("to_id").cast("int").alias("to_id"),
    F.col("scheduled_time").alias("scheduled_time"),
    F.col("actual_time").alias("actual_time"),
    F.round("delay_minutes", 2).alias("delay_minutes"),
    F.trim(F.col("status")).alias("status"),
    F.trim(F.col("line")).alias("line"),
    F.trim(F.col("type")).alias("type"),
)

print(f"Bronze row count: {df_trips.count()}")
print("\nCleaned schema:")
df_trips_cleaned.printSchema()

display(df_trips_cleaned.limit(10))

In [0]:
# Normalize line names: collapse internal whitespace, then map to canonical names
# NJ Transit abbreviations → full names matching the lines reference table
# Amtrak variants → standardized title-case names

line_mapping = {
    # --- NJ Transit lines (abbreviations → canonical) ---
    "Atl. City Line": "Atlantic City Line",
    "Bergen Co. Line": "Bergen County Line",
    "Gladstone Branch": "Gladstone Branch",
    "Main Line": "Main Line",
    "Meadowlands Rail": "Meadowlands Line",
    "Montclair-Boonton": "Montclair-Boonton Line",
    "Morristown Line": "Morristown Line",
    "No Jersey Coast": "NJ Coast Line",
    "Northeast Corrdr": "Northeast Corridor",
    "Pascack Valley": "Pascack Valley Line",
    "Princeton Shuttle": "Princeton Dinky",
    "Raritan Valley": "Raritan Valley Line",
    # --- Amtrak lines (case + variant standardization) ---
    "ACELA EXPRESS": "Acela Express",
    "Acela Express": "Acela Express",
    "AMTRAK": "Amtrak",
    "Amtrak": "Amtrak",
    "AMTRAK REGIONAL": "Amtrak Regional",
    "REGIONAL": "Amtrak Regional",
    "Regional": "Amtrak Regional",
    "CARDINAL": "Cardinal",
    "Cardinal / Hoosier State": "Cardinal",
    "CAROLINIAN": "Carolinian",
    "Carolinian / Piedmont": "Carolinian",
    "CRESCENT": "Crescent",
    "Crescent": "Crescent",
    "KEYSTONE": "Keystone",
    "Keystone": "Keystone",
    "PALMETTO": "Palmetto",
    "Palmetto": "Palmetto",
    "PENNSYLVANIAN": "Pennsylvanian",
    "Pennsylvanian": "Pennsylvanian",
    "SILVER METEOR-R": "Silver Meteor",
    "Silver Meteor": "Silver Meteor",
    "SILVER STAR -R": "Silver Star",
    "Silver Service / Palmetto": "Silver Service / Palmetto",
    "VERMONTER -R": "Vermonter",
}

# Build a Spark map expression for the lookup
map_keys_values = []
for raw, canonical in line_mapping.items():
    map_keys_values.extend([F.lit(raw), F.lit(canonical)])
line_map_expr = F.create_map(*map_keys_values)

# Collapse internal whitespace, then apply mapping (fall back to original if unmapped)
df_trips_cleaned = df_trips_cleaned.withColumn(
    "line",
    F.regexp_replace(F.col("line"), r"\s+", " ")
).withColumn(
    "line",
    F.coalesce(line_map_expr[F.col("line")], F.col("line"))
)

print("Distinct lines after normalization:")
df_trips_cleaned.select("line").distinct().orderBy("line").show(50, truncate=False)

In [0]:
# Drop completely empty rows (25K rows where status, from, to, line, type are all null)
before_count = df_trips_cleaned.count()
df_trips_cleaned = df_trips_cleaned.filter(
    F.col("status").isNotNull()
    | F.col("from").isNotNull()
    | F.col("to").isNotNull()
    | F.col("line").isNotNull()
    | F.col("type").isNotNull()
)
after_count = df_trips_cleaned.count()
print(f"Dropped {before_count - after_count} empty rows ({before_count} → {after_count})")

# Standardize status to lowercase and add derived flags
df_trips_cleaned = df_trips_cleaned.withColumn(
    "status", F.lower(F.trim(F.col("status")))
).withColumn(
    "is_cancelled", F.col("status") == "cancelled"
)

# For cancelled trips, delay_minutes should be NULL (trip never ran)
df_trips_cleaned = df_trips_cleaned.withColumn(
    "delay_minutes",
    F.when(F.col("is_cancelled"), F.lit(None).cast("double")).otherwise(F.col("delay_minutes"))
)

print("\nStatus distribution after cleaning:")
df_trips_cleaned.groupBy("status").count().orderBy("count", ascending=False).show()

print("\nNull counts for cancelled trips:")
df_trips_cleaned.filter(F.col("is_cancelled")).select(
    F.sum(F.col("delay_minutes").isNull().cast("int")).alias("null_delay"),
    F.sum(F.col("scheduled_time").isNull().cast("int")).alias("null_scheduled"),
    F.count(F.lit(1)).alias("total_cancelled")
).show()

In [0]:
# Write cleaned transit_trips to silver layer
df_trips_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_silver.transit_trips")

print("Silver table written: workspace.nj_transit_silver.transit_trips")

silver_count = spark.table("workspace.nj_transit_silver.transit_trips").count()
print(f"Row count: {silver_count}")

print("\nSample rows:")
display(spark.table("workspace.nj_transit_silver.transit_trips").limit(10))

In [0]:
# --- Clean stations table ---

df_stations = spark.table("workspace.nj_transit_bronze.stations")

# Drop empty/useless columns: SE_ANNO_CAD_DATA (all empty), LAYER (all "STATIONS")
cols_to_drop = ["SE_ANNO_CAD_DATA", "LAYER"]
df_stations_cleaned = df_stations.drop(*cols_to_drop)

# Trim whitespace from all string columns
string_cols = [f.name for f in df_stations_cleaned.schema.fields if f.dataType.simpleString() == "string"]
for c in string_cols:
    df_stations_cleaned = df_stations_cleaned.withColumn(c, F.trim(F.col(c)))

# Normalize LINE_NAME to match canonical names used in transit_trips
station_line_mapping = {
    "North Jersey Coast Line": "NJ Coast Line",
    "Sports Line": "Meadowlands Line",
}
station_map_keys_values = []
for raw, canonical in station_line_mapping.items():
    station_map_keys_values.extend([F.lit(raw), F.lit(canonical)])
station_line_map = F.create_map(*station_map_keys_values)

df_stations_cleaned = df_stations_cleaned.withColumn(
    "LINE_NAME",
    F.coalesce(station_line_map[F.col("LINE_NAME")], F.col("LINE_NAME"))
)

# Standardize ADA to lowercase (YES/NO → yes/no)
df_stations_cleaned = df_stations_cleaned.withColumn(
    "ADA", F.lower(F.col("ADA"))
)

print(f"Stations row count: {df_stations_cleaned.count()}")
print("\nCleaned schema:")
df_stations_cleaned.printSchema()

print("\nDistinct LINE_NAME after normalization:")
df_stations_cleaned.select("LINE_NAME").distinct().orderBy("LINE_NAME").show(20, truncate=False)

print("\nSample rows:")
display(df_stations_cleaned.limit(10))

In [0]:
# Write cleaned stations to silver layer
df_stations_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_silver.stations")

print("Silver table written: workspace.nj_transit_silver.stations")
silver_stations_count = spark.table("workspace.nj_transit_silver.stations").count()
print(f"Row count: {silver_stations_count}")

# --- Write lines table to silver ---
# Keep only line metadata, keyed by LINE_CODE. Geometry (geometry_type, coordinates)
# lives in workspace.nj_transit_silver.line_coordinates, the one-to-many points table.
df_lines = spark.table("workspace.nj_transit_bronze.lines")

df_lines_cleaned = df_lines.drop("OBJECTID", "geometry_type", "coordinates")

# Deduplicate on LINE_CODE in case bronze is ever re-imported with duplicate rows
# (bronze was previously triplicated: 42 rows → 14)
df_lines_cleaned = df_lines_cleaned.dropDuplicates(["LINE_CODE"])

# Drop the old table first — a plain overwrite write keeps the previous schema
# and re-adds the dropped columns as NULLs
spark.sql("DROP TABLE IF EXISTS workspace.nj_transit_silver.lines")

df_lines_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_silver.lines")

print(f"\nSilver table written: workspace.nj_transit_silver.lines")
silver_lines_count = spark.table("workspace.nj_transit_silver.lines").count()
print(f"Row count: {silver_lines_count}")

print("\nSample stations:")
display(spark.table("workspace.nj_transit_silver.stations").limit(5))

print("\nSilver lines (line metadata only; geometry in line_coordinates):")
display(spark.table("workspace.nj_transit_silver.lines"))

In [0]:
# --- Explode line coordinates for Tableau map visualization ---
# Tableau needs individual lat/lon rows to plot line geometries
# Parse the GeoJSON coordinate strings into individual points

import json

df_lines = spark.table("workspace.nj_transit_bronze.lines")
rows = df_lines.collect()

coord_records = []
for row in rows:
    coords = json.loads(row["coordinates"])
    geom_type = row["geometry_type"]

    if geom_type == "LineString":
        # coords = [[lon, lat], [lon, lat], ...]
        for pt_idx, (lon, lat) in enumerate(coords):
            coord_records.append((row["LINE_CODE"], row["LINE_NAME"], 0, pt_idx, float(lon), float(lat)))
    elif geom_type == "MultiLineString":
        # coords = [[[lon, lat], ...], [[lon, lat], ...]]
        for seg_idx, segment in enumerate(coords):
            for pt_idx, (lon, lat) in enumerate(segment):
                coord_records.append((row["LINE_CODE"], row["LINE_NAME"], seg_idx, pt_idx, float(lon), float(lat)))

print(f"Total coordinate points: {len(coord_records)}")

df_line_coords = spark.createDataFrame(
    coord_records,
    ["line_code", "line_name", "segment_index", "point_index", "longitude", "latitude"]
)

df_line_coords.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_silver.line_coordinates")